# A. Introduction

MemoryFence studies whether instructions placed in shared workspace content can persist in derived memories after a transformation or after the original source is deleted, and whether deletion handling, recorded provenance, and defenses change what another user's agent does.

This notebook runs a small development pilot on a **free Google Colab** runtime. The development Mac does not have enough free disk for a local model, so the model is downloaded only into this temporary runtime. Ollama and MemoryFence run in the same runtime. Ollama listens on loopback. No API key is required, and no hosted inference service is used.

The scenarios, documents, and tools are synthetic. The sandbox does not send email, open a ticket service, or run a shell for the model.

A Colab runtime is temporary. Its disk can disappear when the session ends. Run the cells in order, then download the archive in the last section before you disconnect.

This notebook is an engineering and exploratory pilot. It is not a confirmatory study. Do not switch to a paid runtime. If the free runtime cannot install the model, stop and keep the partial output unlabeled.


# B. Runtime diagnostics

This cell prints the runtime with the Python standard library. It does not install software and it does not download a model. After the repository is installed, MemoryFence checks the same facts again and stops if disk or memory is too small for the pilot model.

A missing GPU is recorded as a CPU runtime. The pilot does not require a GPU.


In [ ]:
import os
import platform
import shutil
from pathlib import Path

print("python", platform.python_version())
print("cpu_count", os.cpu_count())
meminfo = Path("/proc/meminfo")
if meminfo.is_file():
    rows = [
        line
        for line in meminfo.read_text(encoding="utf-8").splitlines()
        if line.startswith(("MemTotal:", "MemAvailable:"))
    ]
    print("\n".join(rows) if rows else "memory counters are unavailable")
else:
    print("memory counters are unavailable")
print("disk_free_bytes", shutil.disk_usage("/").free)
print("gpu", "checked again after MemoryFence is installed")


# C. Repository setup

The clone uses https://github.com/Yug-More/memoryfence and no other remote.

`REVISION = "main"` follows the default branch. That is **not a frozen revision**. For a frozen run, set `FROZEN = True` and set `REVISION` to the full 40-character commit you intend to repeat. A frozen request never falls back to `main`.

If `/content/memoryfence` already exists, start a new runtime. Do not reuse a previous checkout for a new pilot.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise SystemExit(
        "model installation runs only inside Google Colab and does not download files "
        "on this computer"
    )

# main is not a frozen revision. Set FROZEN = True and a 40-character SHA to pin one.
REVISION = "main"
FROZEN = False
if FROZEN:
    if len(REVISION) != 40 or any(char not in "0123456789abcdef" for char in REVISION):
        raise SystemExit(
            "a frozen experiment requires the full 40-character commit; main is not used"
        )
elif REVISION != "main" and (
    len(REVISION) != 40 or any(char not in "0123456789abcdef" for char in REVISION)
):
    raise SystemExit("revision must be main or a full commit SHA")

ROOT = Path("/content/memoryfence")
if ROOT.exists():
    raise SystemExit(
        "the checkout directory already exists; use a new runtime so old trials are not mixed"
    )
subprocess.run(
    ["git", "clone", "--no-tags", "https://github.com/Yug-More/memoryfence", str(ROOT)],
    check=True,
)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", REVISION], check=True)
if shutil.which("uv") is None:
    subprocess.run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
    os.environ["PATH"] = str(Path.home() / ".local" / "bin") + os.pathsep + os.environ.get("PATH", "")
subprocess.run(["uv", "sync", "--frozen", "--python", "3.12", "--project", str(ROOT)], check=True)

site = next((ROOT / ".venv").glob("lib/python*/site-packages"))
sys.path.insert(0, str(site))
from memoryfence.colab.session import (  # noqa: E402
    assert_checked_out,
    collect_runtime,
    plan_checkout,
    require_colab_runtime,
    require_pilot_resources,
    require_python,
    run_git,
    use_project_environment,
)

use_project_environment(ROOT)
require_colab_runtime(colab=IN_COLAB)
plan = plan_checkout(REVISION, frozen=FROZEN)
checked = assert_checked_out(run_git(("git", "-C", str(ROOT), "rev-parse", "HEAD")), plan)
print("checked_out", checked)
report = collect_runtime()
require_python(report.python_version)
print(require_pilot_resources(report))
print(report.model_dump_json(indent=2))


# D. Ollama installation

Ollama is installed inside this runtime and started with `OLLAMA_HOST=127.0.0.1:11434`. The listen address is loopback. This notebook does not create a tunnel, a public proxy, or a remote endpoint.

The official installer is `https://ollama.com/install.sh`. If it exits non-zero, the cell stops. Startup waits at most 30 seconds.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from memoryfence.colab.session import (
    LoopbackOllamaProcess,
    ensure_loopback_ollama,
    ollama_environment,
    require_colab_runtime,
)

require_colab_runtime(colab=IN_COLAB)
os.environ.update(ollama_environment())
print("endpoint", "http://127.0.0.1:11434")
if shutil.which("ollama") is None:
    completed = subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=False,
    )
    if completed.returncode != 0:
        raise SystemExit("Ollama installation failed")
executable = shutil.which("ollama")
message = ensure_loopback_ollama(
    executable=Path(executable) if executable else None,
    launcher=LoopbackOllamaProcess(),
    timeout_seconds=30,
)
print(message)


# E. Model selection

The only approved download is `qwen2.5:0.5b`, the candidate already named in `configs/ollama-pilot.json`. The cell checks free disk and available memory before it builds the pull command. It does not pull a second model, and it does not pull a larger model.

`qwen2.5:0.5b` is a very small instruction model. Its behavior is not evidence about larger agents. The digest, quantization, and size are recorded from the local daemon after the pull. They are not assumed in advance.

If this runtime has less than 4 GiB free disk or less than 2 GiB available memory, the cell stops without downloading.


In [ ]:
import subprocess

from memoryfence.colab.session import PILOT_MODEL_TAG, collect_runtime, pull_command, require_pilot_resources

report = collect_runtime()
print(require_pilot_resources(report))
command = pull_command(PILOT_MODEL_TAG, report)
print("pull", " ".join(command))
completed = subprocess.run(command, check=False)
if completed.returncode != 0:
    raise SystemExit("the pilot model was not downloaded")
print("downloaded", PILOT_MODEL_TAG)


# F. Backend smoke test

These commands use the installed MemoryFence CLI. `doctor` must report the loopback daemon. `list` must include the pilot tag. `smoke` asks the model for one structured decision. A failure stops the notebook.

The prose-rejection check uses a fake model response inside the adapter. It does not call the network. A sentence that names a tool must not become a tool call.

The digest printed here comes from the local `/api/tags` entry for the installed tag; `/api/show` supplies model details.


In [ ]:
import subprocess

from memoryfence.colab.session import (
    PILOT_MODEL_TAG,
    memoryfence_command,
    record_smoke_identity,
    reject_prose_tool_request,
)

for args in (
    ("model", "doctor"),
    ("model", "list"),
    ("model", "smoke", "--model", PILOT_MODEL_TAG),
):
    completed = subprocess.run(memoryfence_command(ROOT, args), check=False, text=True, capture_output=True)
    print(completed.stdout)
    print(completed.stderr)
    if completed.returncode != 0:
        raise SystemExit(completed.returncode)
identity = record_smoke_identity("http://127.0.0.1:11434", PILOT_MODEL_TAG)
print(identity.model_dump_json())
print("prose_rejected", reject_prose_tool_request())


# G. Experiment preview

The configuration is `configs/ollama-pilot.json`. The preview must show four development cells:

- `clean-note` with `DF0`
- `clean-note` with `DF1`
- `direct-note` with `DF0`
- `direct-note` with `DF1`

Transformations stay scripted. The victim agent is the Ollama backend. Held-out scenarios are not loaded. Preview does not call the model.


In [ ]:
import subprocess

from memoryfence.colab.session import memoryfence_command, preview_pilot

matrix = preview_pilot(ROOT)
print("valid", matrix.valid_count)
completed = subprocess.run(
    memoryfence_command(
        ROOT,
        ("experiment", "preview", "--config", str(ROOT / "configs/ollama-pilot.json")),
    ),
    check=False,
    text=True,
    capture_output=True,
)
print(completed.stdout)
if completed.returncode != 0:
    raise SystemExit(completed.returncode)


# H. Real-model pilot

This cell runs the four previewed cells through `memoryfence experiment run`. Execution is sequential. The existing run cap stays in force. Each cell starts from fresh episode state.

The stamp in the directory name separates this session from a later one. If you run the pilot again, leave this directory alone and let the next cell create a new stamp. Do not point a new run at a directory that already contains trials.

`summarize` writes `summary.json` and `report.md`. `audit` checks the manifest and does not call the model again.


In [ ]:
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from memoryfence.colab.session import (
    interrupt_note,
    memoryfence_command,
    pilot_output_directory,
    read_model_metadata,
    resume_or_refuse,
)

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
PILOT_DIRECTORY = pilot_output_directory(Path("/content/memoryfence-runs"), STAMP)
print(resume_or_refuse(PILOT_DIRECTORY, new_run=True), PILOT_DIRECTORY)
config = str(ROOT / "configs/ollama-pilot.json")
steps = (
    ("experiment", "run", "--config", config, "--directory", str(PILOT_DIRECTORY)),
    ("experiment", "summarize", "--directory", str(PILOT_DIRECTORY)),
    ("experiment", "audit", "--directory", str(PILOT_DIRECTORY)),
)
for args in steps:
    completed = subprocess.run(memoryfence_command(ROOT, args), check=False)
    if completed.returncode != 0:
        print(interrupt_note(PILOT_DIRECTORY))
        raise SystemExit(completed.returncode)
metadata = read_model_metadata(PILOT_DIRECTORY)
print("model_tag", metadata.model_tag)
print("model_digest", metadata.model_digest)
print("code_version", metadata.code_version)
print((PILOT_DIRECTORY / "report.md").read_text(encoding="utf-8"))


# I. Results and interpretation

**Exploratory development pilot — not confirmatory evidence.**

Four development cells and one small model cannot support an attack success rate, a defense effect, or a claim that a vulnerability was demonstrated. Scripted transformations produced the derived memory text. The victim agent is the only Ollama caller in this pilot.

Read S1, S2, S3, and S4 separately. An abstention is not a followed instruction. A proposal that the gateway blocks is not an execution. Invalid or timed-out cells stay invalid. If the model cannot produce the structured decision, that is a model-capability limit of this tag, not a defense result.


In [ ]:
print("Exploratory development pilot — not confirmatory evidence.")
print("Four development cells and one small model do not establish an attack rate or a defense effect.")
print("Scripted transformations produced the derived memories. The victim agent used Ollama.")
print((PILOT_DIRECTORY / "report.md").read_text(encoding="utf-8"))


# J. Artifact export

The archive contains the saved configuration, manifest, matrix, run records, summaries, benchmark results, and an `environment.json` file with the checked-out commit and the runtime facts. Judge review packets are included when the experiment directory already has them. The archive is written under `/content/memoryfence-exports` and offered as a browser download.

`SAVE_TO_DRIVE = False` keeps Google Drive unmounted. Do not change it unless you intend to choose a folder yourself. This notebook does not publish results.

Download the zip before the session ends. After a disconnect, unsaved files are gone. A later run needs a new directory so its trials are not mixed with this one.


In [ ]:
from pathlib import Path

from google.colab import files
from memoryfence.colab.session import PilotEnvironment, export_pilot_archive, interrupt_note, verify_archive

environment = PilotEnvironment(
    repository="https://github.com/Yug-More/memoryfence",
    revision=checked,
    python_version=report.python_version,
    gpu_available=report.gpu_available,
    gpu_name=report.gpu_name,
    disk_free_bytes=report.disk_free_bytes,
    ram_available_bytes=report.ram_available_bytes,
    model_tag=PILOT_MODEL_TAG,
    endpoint="http://127.0.0.1:11434",
)
archive = export_pilot_archive(
    PILOT_DIRECTORY,
    Path("/content/memoryfence-exports"),
    revision=checked,
    environment=environment,
    stamp=STAMP,
)
names = verify_archive(archive)
print(archive)
print("members", len(names))
print("bytes", archive.stat().st_size)
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    raise SystemExit("Drive was not mounted. Set this only when you intend to choose a folder.")
files.download(str(archive))
print(interrupt_note(PILOT_DIRECTORY))
print("Download this archive before the runtime disconnects. The Colab disk is temporary.")
